In [ ]:
"""
Análisis Tecno-Económico del Agregador en España
Cuantificación Geométrica de la Flexibilidad Agregada: El Zonotopo

Este script visualiza la capacidad de flexibilidad conjunta de múltiples Recursos 
Energéticos Distribuidos (DERs) mediante el cálculo de su Zonotopo. Matemáticamente, 
esto se obtiene aplicando la Suma de Minkowski a los vectores de flexibilidad 
individuales de cada recurso, acotados por sus restricciones operativas, definiendo 
así la envolvente convexa o "Planta de Energía Virtual".

Autor: Alberto Zafra Muñoz
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import itertools
from typing import Tuple, List

# ==========================================
# 1. MODELADO MATEMÁTICO DEL ZONOTOPO
# ==========================================
def calcular_envolvente_zonotopo(
    centro_operativo: np.ndarray, 
    vectores_generadores: np.ndarray
) -> Tuple[np.ndarray, ConvexHull]:
    """
    Calcula los vértices del zonotopo y su envolvente convexa.
    
    El zonotopo representa el espacio de estados factible de la flexibilidad agregada,
    calculado como la combinación lineal de los vectores generadores multiplicados por 
    coeficientes en el intervalo [-1, 1].

    Parámetros:
        centro_operativo (np.ndarray): Coordenadas del punto base (baseline).
        vectores_generadores (np.ndarray): Matriz con los vectores de flexibilidad de cada DER.

    Retorna:
        Tuple: (Matriz de todos los puntos generados, Objeto ConvexHull de la frontera)
    """
    num_generadores = len(vectores_generadores)
    
    # Generamos todas las permutaciones posibles de los coeficientes extremos [-1, 1]
    coeficientes_extremos = list(itertools.product([-1, 1], repeat=num_generadores))
    
    # Calculamos las coordenadas resultantes para cada combinación lineal
    puntos_espacio = []
    for coef in coeficientes_extremos:
        desplazamiento = sum(coef[i] * vectores_generadores[i] for i in range(num_generadores))
        puntos_espacio.append(centro_operativo + desplazamiento)
        
    puntos_array = np.array(puntos_espacio)
    
    # Calculamos la envolvente convexa para extraer el contorno exterior perimetral
    envolvente = ConvexHull(puntos_array)
    
    return puntos_array, envolvente

# ==========================================
# 2. REPRESENTACIÓN GRÁFICA ACADÉMICA
# ==========================================
def visualizar_flexibilidad_agregada(
    centro: np.ndarray, 
    generadores: np.ndarray, 
    puntos: np.ndarray, 
    hull: ConvexHull,
    nombres_der: List[str],
    colores_der: List[str]
):
    """
    Genera la representación gráfica bidimensional del zonotopo y sus vectores base.
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    plt.style.use('seaborn-v0_8-whitegrid')

    # 1. Trazado del Polítopo (La "Batería Virtual")
    # hull.vertices contiene los índices de los puntos que forman el perímetro
    ax.fill(puntos[hull.vertices, 0], puntos[hull.vertices, 1], 
            color='#9467bd', alpha=0.3, edgecolor='#4b0082', linewidth=2, 
            label='Zonotopo Agregado (Planta de Energía Virtual)')

    # 2. Trazado de los Vectores Generadores Individuales
    for i, vector in enumerate(generadores):
        ax.arrow(
            centro[0], centro[1], vector[0], vector[1], 
            head_width=0.3, head_length=0.4,
            fc=colores_der[i], ec=colores_der[i], length_includes_head=True,
            linewidth=2.5, zorder=4, label=f'Vector Flexibilidad {nombres_der[i]}'
        )

    # 3. Marcador del Centro de Operación
    ax.plot(centro[0], centro[1], 'ko', markersize=8, zorder=5, 
            label='Punto Operativo Base (Baseline)')

    # 4. Formateo y Estilos de la Figura
    plt.title('Cuantificación Geométrica de Flexibilidad: El Zonotopo', 
              fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Capacidad de Potencia: Horizonte $t$ ($P_t$)', fontsize=12)
    plt.ylabel('Capacidad de Potencia: Horizonte $t+1$ ($P_{t+1}$)', fontsize=12)

    # El aspecto 'equal' es crucial para no deformar la percepción geométrica de los vectores
    ax.set_aspect('equal')
    plt.grid(True, linestyle='--', alpha=0.7)

    # Configuración de la leyenda ubicándola fuera del gráfico para no tapar el zonotopo
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1.05, 1), 
              frameon=True, fontsize=10, shadow=True)

    plt.tight_layout()
    plt.show()

# ==========================================
# 3. BLOQUE PRINCIPAL DE EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    # Parametrización del Sistema (4 Recursos Energéticos Distribuidos)
    # Vectores definidos en el plano bidimensional de potencias o temporal
    VECTORES_DER = np.array([
        [ 3.0,  1.0],  # DER 1: Sistema de Almacenamiento (BESS 1)
        [ 1.0,  4.0],  # DER 2: Vehículo Eléctrico (EV)
        [-2.0,  2.5],  # DER 3: Bomba de Calor (HVAC)
        [ 1.5, -1.0]   # DER 4: Sistema de Almacenamiento (BESS 2)
    ])
    
    PUNTO_BASE = np.array([5.0, 5.0])
    
    NOMBRES_RECURSOS = ['BESS 1', 'EV', 'HVAC', 'BESS 2']
    COLORES_RECURSOS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    print("Calculando la suma de Minkowski (Zonotopo) del clúster de DERs...")
    matriz_puntos, frontera_convexa = calcular_envolvente_zonotopo(PUNTO_BASE, VECTORES_DER)
    
    print("Generando modelo visual...")
    visualizar_flexibilidad_agregada(
        PUNTO_BASE, 
        VECTORES_DER, 
        matriz_puntos, 
        frontera_convexa, 
        NOMBRES_RECURSOS, 
        COLORES_RECURSOS
    )